In [37]:
import pandas as pd
import numpy as np

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

def tok(x):
    return list(x.lower())

train.head()

,SampleID,Text,RevisedText
0,747,She forgot her umbrella it started to rain.,"She forgot her umbrella, and then it started t..."
1,1382,He could have bought the house if he has enoug...,He could have bought the house if he had enoug...
2,241,I have a meeting with a principal of a school.,I have a meeting with the principal of the sch...
3,1647,26. You wanna grab some dinner later?,26. Do you want to have dinner later?
4,1461,The bicycle is more cheaper than the car.,The bicycle is cheaper than the car.


In [38]:
import torch
import torch.nn as nn

num_epochs = 20
batch_size = 32
lr = 1e-3

device = torch.device("cuda")
device

device(type='cuda')

In [39]:
from torch.utils.data import Dataset, DataLoader
from collections import Counter

PAD, SOS, EOS = 0, 1, 2

cnt = Counter()
for t in train["Text"]:
    cnt.update(tok(t))
for t in train["RevisedText"]:
    cnt.update(tok(t))

vocab = {"<PAD>":0,"<SOS>":1,"<EOS>":2}
for w,f in cnt.items():
    vocab[w] = len(vocab)

inv_vocab = {i:w for w,i in vocab.items()}

def enc(x, max_len=100):
    x = [vocab.get(w, UNK) for w in tok(x)]
    return [SOS] + x[:max_len-2] + [EOS]

In [40]:
class DS(Dataset):
    def __init__(self, df): self.df = df
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        return (
            torch.tensor(enc(self.df.iloc[i]["Text"])),
            torch.tensor(enc(self.df.iloc[i]["RevisedText"]))
        )

def collate(batch):
    s,t = zip(*batch)
    s = nn.utils.rnn.pad_sequence(s,batch_first=True,padding_value=PAD)
    t = nn.utils.rnn.pad_sequence(t,batch_first=True,padding_value=PAD)
    return s,t

loader = DataLoader(DS(train), batch_size=32, shuffle=True, collate_fn=collate)

In [41]:
class Seq2Seq(nn.Module):
    def __init__(self,v):
        super().__init__()
        self.emb = nn.Embedding(v,128,padding_idx=PAD)
        self.enc = nn.LSTM(128,256,batch_first=True)
        self.dec = nn.LSTM(128,256,batch_first=True)
        self.fc = nn.Linear(256,v)

    def forward(self,s,t):
        _, (h,c) = self.enc(self.emb(s))

        inp = t[:,0].unsqueeze(1)
        out = torch.zeros(s.size(0),t.size(1),len(vocab)).to(s.device)

        for i in range(1,t.size(1)):
            x = self.emb(inp)
            o,(h,c) = self.dec(x,(h,c))
            o = self.fc(o.squeeze(1))
            out[:,i] = o
            inp = t[:,i].unsqueeze(1)

        return out

model = Seq2Seq(len(vocab)).to(device)
opt = torch.optim.Adam(model.parameters(),1e-3)
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD)

model

Seq2Seq(
  (emb): Embedding(58, 128, padding_idx=0)
  (enc): LSTM(128, 256, batch_first=True)
  (dec): LSTM(128, 256, batch_first=True)
  (fc): Linear(in_features=256, out_features=58, bias=True)
)

In [42]:
for i in range(1, num_epochs+1):
    model.train()
    total_loss = 0
    for s,t in loader:
        s,t = s.to(device),t.to(device)
        opt.zero_grad()
        o = model(s,t)
        loss = loss_fn(o[:,1:].reshape(-1,len(vocab)), t[:,1:].reshape(-1))
        loss.backward()
        opt.step()
        total_loss += loss.item()
    print("Epoch", i)
    print("Loss", total_loss/len(loader))

Epoch 1
Loss 2.7585933301963057
Epoch 2
Loss 2.1622332311144064
Epoch 3
Loss 1.990843889760036
Epoch 4
Loss 1.8751962231654746
Epoch 5
Loss 1.7915470670251286
Epoch 6
Loss 1.718123658030641
Epoch 7
Loss 1.654716131733913
Epoch 8
Loss 1.5980933063170488
Epoch 9
Loss 1.5484204292297363
Epoch 10
Loss 1.5014615316017002
Epoch 11
Loss 1.4623962313521142
Epoch 12
Loss 1.4213702912424124
Epoch 13
Loss 1.3845203811047124
Epoch 14
Loss 1.3496785794987398
Epoch 15
Loss 1.3197381379557591
Epoch 16
Loss 1.28731449211345
Epoch 17
Loss 1.2607519883735507
Epoch 18
Loss 1.2318349772808599
Epoch 19
Loss 1.2079679264741785
Epoch 20
Loss 1.1823257745481004


In [43]:
from tqdm import tqdm

def predict(text, max_len=40):
    model.eval()
    s = torch.tensor([enc(text)]).to(device)

    _, (h,c) = model.enc(model.emb(s))
    inp = torch.tensor([[SOS]]).to(device)

    res = []

    for _ in range(max_len):
        x = model.emb(inp)
        o,(h,c) = model.dec(x,(h,c))
        o = model.fc(o.squeeze(1))
        p = o.argmax(1).item()

        if p == EOS: break
        res.append(inv_vocab.get(p,""))
        inp = torch.tensor([[p]]).to(device)

    return "".join(res)

preds = []
for _, row in tqdm(test.iterrows()):
    preds.append(predict(row["Text"]))

preds[:5]

404it [00:17, 23.76it/s]


['the company is street to the store.',
 'the company is street to the store were ',
 'the company is street to the store.',
 'the company is street to the store were ',
 'the company is street to the store were ']

In [44]:
submission = pd.DataFrame({
    "DatapointID": test["SampleID"],
    "RevisedText": preds
})

submission

,DatapointID,RevisedText
0,1557,the company is street to the store.
1,527,the company is street to the store were
2,394,the company is street to the store.
3,1790,the company is street to the store were
4,434,the company is street to the store were
...,...,...
399,1113,the company is street to the store were
400,694,the company is street to the store.
401,1495,the company is street in the park.
402,922,the company is street to the store.


In [45]:
test

,SampleID,Text
0,1557,He didn't eat no breakfast this morning.
1,527,"29. When I was young, I used to played soccer ..."
2,394,The car is parked in front of the house.
3,1790,"They're house is beautiful, but it's location ..."
4,434,I can't believe she ate the whole cake herself!
...,...,...
399,1113,This is the most hardest puzzle I've ever solved.
400,694,she loves reading books by jane austen.
401,1495,The book it's on the shelf.
402,922,The dog was walked by Tim in the park.


In [46]:
submission.to_csv("submission.csv", index=False)